In [ ]:
#Lab Task 1 – A 2D Grid Vacuum World with a Simple Reflex Agent

In [22]:
import random

class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)

        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []

        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")

        return moves

    def apply_action(self, action):
        r, c = self.position

        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)

In [23]:
def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    return random.choice(env.valid_moves())

In [24]:
def run_agent(env, decide, max_steps=200):
    steps = 0

    for _ in range(max_steps):
        percept = env.get_percept()

        if env.is_clean():
            break

        action = decide(percept, env)
        env.apply_action(action)
        steps += 1

    return steps


env = GridEnvironment(rows=4, cols=4, dirt_probability=0.5, seed=11)
steps = run_agent(env, simple_reflex_agent)

print(f"Simple reflex (random) steps: {steps}")

Simple reflex (random) steps: 48


In [ ]:
#Lab Task 2 – A Model-Based Agent That Remembers Where It Has Been

In [25]:
def neighbor_position(pos, action):
    r, c = pos
    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c)
    }[action]


class ModelBasedAgent:
    def __init__(self):
        self.visited = set()

    def act(self, percept, env):
        self.visited.add(percept["position"])

        if percept["is_dirty"]:
            return "Suck"

        moves = env.valid_moves()
        here = percept["position"]

        unvisited = [
            m for m in moves
            if neighbor_position(here, m) not in self.visited
        ]

        if unvisited:
            return random.choice(unvisited)

        return random.choice(moves)

In [26]:

import random
import copy

# Grid Environment
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)

        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {
            "position": (r, c),
            "is_dirty": self.dirt[r][c]
        }

    def valid_moves(self):
        r, c = self.position
        moves = []

        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")

        return moves

    def apply_action(self, action):
        r, c = self.position

        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


# Simple Reflex Agent
def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    return random.choice(env.valid_moves())


# Find Neighbor Position
def neighbor_position(pos, action):
    r, c = pos
    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c)
    }[action]


# Model-Based Agent
class ModelBasedAgent:
    def __init__(self):
        self.visited = set()

    def act(self, percept, env):
        self.visited.add(percept["position"])

        if percept["is_dirty"]:
            return "Suck"

        moves = env.valid_moves()
        here = percept["position"]

        unvisited = [
            m for m in moves
            if neighbor_position(here, m) not in self.visited
        ]

        if unvisited:
            return random.choice(unvisited)

        return random.choice(moves)


# Run Agent
def run_agent(env, decide, max_steps=200):
    steps = 0

    for _ in range(max_steps):
        percept = env.get_percept()

        if env.is_clean():
            break

        action = decide(percept, env)
        env.apply_action(action)
        steps += 1

    return steps


# Compare Both Agents on the Same Starting Grid
random.seed(11)

env1 = GridEnvironment(rows=4, cols=4, dirt_probability=0.5)
dirt_copy = copy.deepcopy(env1.dirt)

steps1 = run_agent(env1, simple_reflex_agent)
print("Simple reflex (random) steps:", steps1)
print("Simple reflex cleaned all cells:", env1.is_clean())

random.seed(11)

env2 = GridEnvironment(rows=4, cols=4, dirt_probability=0.5)
env2.dirt = copy.deepcopy(dirt_copy)

agent2 = ModelBasedAgent()
steps2 = run_agent(env2, agent2.act)

print("Model-based steps:", steps2)
print("Model-based cleaned all cells:", env2.is_clean())


Simple reflex (random) steps: 48
Simple reflex cleaned all cells: True
Model-based steps: 42
Model-based cleaned all cells: True


In [ ]:
#Lab Task 3 – A Goal-Based Agent That Heads to One Target Room

In [1]:
def goal_based_agent(position, goal):
    r, c = position
    gr, gc = goal

    if r < gr:
        return "Down"
    if r > gr:
        return "Up"
    if c < gc:
        return "Right"
    if c > gc:
        return "Left"

    return "NoOp"

In [4]:

# Grid Environment
class GridEnvironment:
    def __init__(self, rows=4, cols=4):
        self.rows = rows
        self.cols = cols
        self.position = (0, 0)

    def apply_action(self, action):
        r, c = self.position

        if action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)


# Goal-Based Agent
def goal_based_agent(position, goal):
    r, c = position
    gr, gc = goal

    if r < gr:
        return "Down"
    if r > gr:
        return "Up"
    if c < gc:
        return "Right"
    if c > gc:
        return "Left"

    return "NoOp"


# Run 1: Start at (0, 0), goal (3, 2)
env = GridEnvironment(rows=4, cols=4)
env.position = (0, 0)
goal = (3, 2)

print("RUN 1")
print(f"Start position: {env.position}, Goal: {goal}")

for step in range(20):
    if env.position == goal:
        print(f"Reached the goal in {step} steps.")
        break

    action = goal_based_agent(env.position, goal)
    env.apply_action(action)
    print(f"Step {step + 1}: moved {action} -> now at {env.position}")


# Run 2: Start at (2, 3), goal (0, 1)
env = GridEnvironment(rows=4, cols=4)
env.position = (2, 3)
goal = (0, 1)

print("\nRUN 2")
print(f"Start position: {env.position}, Goal: {goal}")

for step in range(20):
    if env.position == goal:
        print(f"Reached the goal in {step} steps.")
        break

    action = goal_based_agent(env.position, goal)
    env.apply_action(action)
    print(f"Step {step + 1}: moved {action} -> now at {env.position}")


RUN 1
Start position: (0, 0), Goal: (3, 2)
Step 1: moved Down -> now at (1, 0)
Step 2: moved Down -> now at (2, 0)
Step 3: moved Down -> now at (3, 0)
Step 4: moved Right -> now at (3, 1)
Step 5: moved Right -> now at (3, 2)
Reached the goal in 5 steps.

RUN 2
Start position: (2, 3), Goal: (0, 1)
Step 1: moved Up -> now at (1, 3)
Step 2: moved Up -> now at (0, 3)
Step 3: moved Left -> now at (0, 2)
Step 4: moved Left -> now at (0, 1)
Reached the goal in 4 steps.


In [ ]:
#Lab Task 5 – Compare Agents Fairly: Many Layouts, Not One

In [1]:

def steps_per_layout(env_class, make_decide, n_layouts=200, rows=4, cols=4, dirt_probability=0.5):
    results = []

    for i in range(n_layouts):
        # Create the environment with a specific seed
        env = env_class(rows, cols, dirt_probability, seed=i)

        # Create a fresh decision function for each layout
        decide = make_decide()

        # Run the agent and record its steps
        steps = run_agent(env, decide, max_steps=1000)
        results.append(steps)

    return results


In [5]:

import random
import statistics

# 1. Environment
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)

        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []

        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")

        return moves

    def apply_action(self, action):
        r, c = self.position

        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


# 2. Simple Reflex Agent
def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    return random.choice(env.valid_moves())


# 3. Helper Function
def run_agent(env, decide, max_steps=1000):
    steps = 0

    for _ in range(max_steps):
        if env.is_clean():
            break

        percept = env.get_percept()
        action = decide(percept, env)
        env.apply_action(action)
        steps += 1

    return steps


# 4. Model-Based Agent
def neighbor_position(pos, action):
    r, c = pos

    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c)
    }[action]


class ModelBasedAgent:
    def __init__(self):
        self.visited = set()

    def act(self, percept, env):
        self.visited.add(percept["position"])

        if percept["is_dirty"]:
            return "Suck"

        moves = env.valid_moves()
        here = percept["position"]

        unvisited = [
            move for move in moves
            if neighbor_position(here, move) not in self.visited
        ]

        if unvisited:
            return random.choice(unvisited)

        return random.choice(moves)


# 5. Task 5: Test Agents on 200 Layouts
def steps_per_layout(
    env_class,
    make_decide,
    n_layouts=200,
    rows=4,
    cols=4,
    dirt_probability=0.5
):
    results = []

    for i in range(n_layouts):
        env = env_class(rows, cols, dirt_probability, seed=i)
        decide = make_decide()

        steps = run_agent(env, decide, max_steps=1000)
        results.append(steps)

    return results


# 6. Compare Both Agents
reflex = steps_per_layout(
    GridEnvironment,
    lambda: simple_reflex_agent
)

model = steps_per_layout(
    GridEnvironment,
    lambda: ModelBasedAgent().act
)

worse = sum(
    model_steps > reflex_steps
    for reflex_steps, model_steps in zip(reflex, model)
)

print("Results over 200 layouts")
print("------------------------")
print("Simple reflex average steps:", round(statistics.mean(reflex), 2))
print("Model-based average steps:", round(statistics.mean(model), 2))
print("Model-based needed more steps on:", worse, "layouts")


Results over 200 layouts
------------------------
Simple reflex average steps: 71.69
Model-based average steps: 29.23
Model-based needed more steps on: 5 layouts


In [ ]:
#Lab Task 6 – Upgrade the Sensors

In [6]:

class SensorUpgradeEnvironment(GridEnvironment):
    def get_percept(self):
        percept = super().get_percept()

        dirty_neighbors = []

        for move in self.valid_moves():
            next_pos = neighbor_position(self.position, move)
            r, c = next_pos

            if self.dirt[r][c]:
                dirty_neighbors.append(move)

        percept["dirty_neighbors"] = dirty_neighbors

        return percept


In [8]:

def sensor_agent(percept, env):
    # 1. Clean the current room if dirty
    if percept["is_dirty"]:
        return "Suck"

    # 2. Move towards a dirty neighbouring room
    if percept["dirty_neighbors"]:
        return random.choice(percept["dirty_neighbors"])

    # 3. Otherwise, move randomly
    return random.choice(env.valid_moves())


In [9]:

env = SensorUpgradeEnvironment(4, 4, 0.5, seed=11)
env.position = (0, 2)

print(env.get_percept())


{'position': (0, 2), 'is_dirty': False, 'dirty_neighbors': ['Right', 'Down']}


In [10]:

sensor = steps_per_layout(
    SensorUpgradeEnvironment,
    lambda: sensor_agent
)

print("Sensor agent average steps:", round(sum(sensor) / len(sensor), 2))


Sensor agent average steps: 30.23


In [11]:

print("Simple reflex average:", round(sum(reflex) / len(reflex), 2))
print("Model-based average:", round(sum(model) / len(model), 2))
print("Sensor agent average:", round(sum(sensor) / len(sensor), 2))


Simple reflex average: 71.69
Model-based average: 29.23
Sensor agent average: 30.23


In [ ]:
#Lab Task 7 – A Utility-Based Agent: When Goals Compete

In [16]:
def distance(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def dirty_rooms(env):
    return [
        (r, c)
        for r in range(env.rows)
        for c in range(env.cols)
        if env.dirt[r][c]
    ]


class ValuedEnvironment(GridEnvironment):

    def __init__(self, rows=6, cols=6,
                 dirt_probability=0.5, seed=None):

        super().__init__(rows, cols, dirt_probability, seed)

        self.value = [
            [random.randint(1, 5) for _ in range(cols)]
            for _ in range(rows)
        ]

        self.collected = 0

    def apply_action(self, action):
        r, c = self.position

        if action == "Suck" and self.dirt[r][c]:
            self.collected += self.value[r][c]

        super().apply_action(action)

In [17]:
def nearest_agent(percept, env):

    if percept["is_dirty"]:
        return "Suck"

    position = percept["position"]
    rooms = dirty_rooms(env)

    if not rooms:
        return "NoOp"

    target = min(
        rooms,
        key=lambda room: distance(position, room)
    )

    return goal_based_agent(position, target)


def utility_agent(percept, env):

    if percept["is_dirty"]:
        return "Suck"

    position = percept["position"]
    rooms = dirty_rooms(env)

    if not rooms:
        return "NoOp"

    target = max(
        rooms,
        key=lambda room:
            env.value[room[0]][room[1]]
            / (distance(position, room) + 1)
    )

    return goal_based_agent(position, target)

In [19]:
import random

# 1. Helper functions
def distance(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def dirty_rooms(env):
    return [
        (r, c)
        for r in range(env.rows)
        for c in range(env.cols)
        if env.dirt[r][c]
    ]


# 2. Goal-based agent from Task 3
def goal_based_agent(position, goal):
    r, c = position
    gr, gc = goal

    if r < gr:
        return "Down"
    elif r > gr:
        return "Up"
    elif c < gc:
        return "Right"
    elif c > gc:
        return "Left"
    else:
        return "NoOp"


# 3. Valued environment
class ValuedEnvironment(GridEnvironment):
    def __init__(self, rows=6, cols=6,
                 dirt_probability=0.5, seed=None):

        super().__init__(rows, cols, dirt_probability, seed)

        self.value = [
            [random.randint(1, 5) for _ in range(cols)]
            for _ in range(rows)
        ]
        self.collected = 0

    def apply_action(self, action):
        r, c = self.position

        if action == "Suck" and self.dirt[r][c]:
            self.collected += self.value[r][c]

        super().apply_action(action)


# 4. Nearest-room agent
def nearest_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"

    position = percept["position"]
    rooms = dirty_rooms(env)

    if not rooms:
        return "NoOp"

    target = min(
        rooms,
        key=lambda room: distance(position, room)
    )

    return goal_based_agent(position, target)


# 5. Utility-based agent
def utility_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"

    position = percept["position"]
    rooms = dirty_rooms(env)

    if not rooms:
        return "NoOp"

    target = max(
        rooms,
        key=lambda room:
            env.value[room[0]][room[1]]
            / (distance(position, room) + 1)
    )

    return goal_based_agent(position, target)


# 6. Self-check
env = ValuedEnvironment(6, 6, 0.0, seed=0)

env.dirt[1][0] = True
env.value[1][0] = 1

env.dirt[0][3] = True
env.value[0][3] = 5

percept = env.get_percept()

print("Self-check:")
print(nearest_agent(percept, env),
      utility_agent(percept, env))


# 7. Compare both agents over 200 layouts
def value_collected(decide, n_layouts=200, steps=20):
    totals = []

    for i in range(n_layouts):
        env = ValuedEnvironment(6, 6, 0.5, seed=i)

        for _ in range(steps):
            action = decide(env.get_percept(), env)
            env.apply_action(action)

        totals.append(env.collected)

    return totals


nearest_scores = value_collected(nearest_agent)
utility_scores = value_collected(utility_agent)

nearest_avg = sum(nearest_scores) / len(nearest_scores)
utility_avg = sum(utility_scores) / len(utility_scores)

wins = sum(
    utility > nearest
    for utility, nearest in zip(utility_scores, nearest_scores)
)

print("\nComparison over 200 layouts:")
print("Nearest Agent Average:", round(nearest_avg, 2))
print("Utility Agent Average:", round(utility_avg, 2))
print("Utility Agent Wins:", wins, "out of 200")

Self-check:
Down Right

Comparison over 200 layouts:
Nearest Agent Average: 26.42
Utility Agent Average: 29.87
Utility Agent Wins: 160 out of 200


In [ ]:
#Lab Task 8 – Guardrails in Code: Least Access, Ask First, Keep a Record

In [27]:
from collections import Counter

RESTRICTED = {(1, 1), (2, 2)}
NEEDS_APPROVAL = {(0, 2), (3, 0)}

def ask_human(position):
    """A simulated person. Return True to approve, False to refuse."""
    return False


def run_guarded(env, decide, max_steps=100):
    log = []

    for step in range(1, max_steps + 1):
        if env.is_clean():
            break

        percept = env.get_percept()
        action = decide(percept, env)
        guarded_step(env, action, step, log)

    return log

In [28]:
def guarded_step(env, action, step, log):
    position = env.position
    outcome = "done"

    # TODO 1: Least access
    if action in ["Up", "Down", "Left", "Right"]:
        target = neighbor_position(position, action)

        if target in RESTRICTED:
            outcome = "blocked"
        else:
            env.apply_action(action)

    # TODO 2: Ask first
    elif action == "Suck" and position in NEEDS_APPROVAL:
        if ask_human(position):
            env.apply_action(action)
            outcome = "approved"
        else:
            outcome = "refused"

    # TODO 3: Apply all other actions
    else:
        env.apply_action(action)

    # TODO 4: Keep a record
    log.append({
        "step": step,
        "position": position,
        "action": action,
        "outcome": outcome
    })

In [29]:
env = GridEnvironment(4, 4, 0.5, seed=11)

log = run_guarded(env, simple_reflex_agent)

assert all(
    entry["position"] not in RESTRICTED
    for entry in log
), "Entered a restricted room!"

print(Counter(entry["outcome"] for entry in log))

for entry in log[:3]:
    print(entry)

Counter({'done': 54, 'refused': 35, 'blocked': 11})
{'step': 1, 'position': (0, 0), 'action': 'Suck', 'outcome': 'done'}
{'step': 2, 'position': (0, 0), 'action': 'Down', 'outcome': 'done'}
{'step': 3, 'position': (1, 0), 'action': 'Down', 'outcome': 'done'}
